# What Keeps a Reddit Rivalry Alive?

We study how negative references between Reddit communities change over time.
Our aim is to distinguish declining negativity while interaction continues from a lack of further observed contact.

## Research questions

1. After repeated negative references, does negativity persist, decline with continued contact, or give way to silence?
2. Are these trajectories associated with audience similarity and negative references in both directions during the baseline period?
3. Are shared negative targets associated with different trajectories?

A hyperlink records a reference from a source community to a target community. A negative label indicates negativity toward the linked post; a nonnegative label includes both neutral and positive references. These observations alone do not establish attacks, friendship, or reconciliation.

## 1. Setup

Paths are relative to the repository root. Raw source files are stored in `data/raw/`.

In [1]:
from pathlib import Path

# Open this notebook from the repository root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Open results.ipynb with readait as the working directory.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"


## 2. Hyperlink data: a first look

The Reddit Hyperlink Network contains separate files for links appearing in post bodies and titles. We inspect the first 1,000 rows of each file, preserving the original values before any cleaning.

Each row records a hyperlink from a post in one subreddit (the **source**) to a post in another subreddit (the **target**).

| Column | Meaning |
|---|---|
| `SOURCE_SUBREDDIT` | Community containing the linking post |
| `TARGET_SUBREDDIT` | Community containing the linked post |
| `POST_ID` | Identifier of the source post |
| `TIMESTAMP` | Publication time of the source post |
| `LINK_SENTIMENT` | −1 for negativity toward the linked post; +1 for neutral or positive references |
| `PROPERTIES` | 86 comma-separated numerical features of the source post |

The release does not contain the target-post identifier or the original text of either post.

In [2]:
import pandas as pd
from IPython.display import display

from src.data.loaders import load_link_sample, load_embedding_sample

body_sample = load_link_sample(RAW_DATA_DIR / "soc-redditHyperlinks-body.tsv")
title_sample = load_link_sample(RAW_DATA_DIR / "soc-redditHyperlinks-title.tsv")

print("Body hyperlinks")
display(body_sample.head())
print("Title hyperlinks")
display(title_sample.head())

Body hyperlinks


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1,"345.0,298.0,0.75652173913,0.0173913043478,0.08..."
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1,"101.0,98.0,0.742574257426,0.019801980198,0.049..."
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1,"85.0,85.0,0.752941176471,0.0235294117647,0.082..."
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1,"1124.0,949.0,0.772241992883,0.0017793594306,0...."
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1,"715.0,622.0,0.777622377622,0.00699300699301,0...."


Title hyperlinks


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,rddtgaming,rddtrust,1u4pzzs,2013-12-31 16:39:18,1,"25.0,23.0,0.76,0.0,0.44,0.12,0.12,4.0,4.0,0.0,..."
1,xboxone,battlefield_4,1u4tmfs,2013-12-31 17:59:11,1,"100.0,88.0,0.78,0.02,0.08,0.13,0.07,16.0,16.0,..."
2,ps4,battlefield_4,1u4tmos,2013-12-31 17:59:40,1,"100.0,88.0,0.78,0.02,0.08,0.13,0.07,16.0,16.0,..."
3,fitnesscirclejerk,leangains,1u50xfs,2013-12-31 19:01:56,1,"49.0,43.0,0.775510204082,0.0,0.265306122449,0...."
4,fitnesscirclejerk,lifeprotips,1u51nps,2013-12-31 21:02:28,1,"14.0,14.0,0.785714285714,0.0,0.428571428571,0...."


In [3]:
sample_overview = pd.DataFrame({
    "body_dtype": body_sample.dtypes.astype(str),
    "title_dtype": title_sample.dtypes.astype(str),
})
display(sample_overview)
print(f"Preview rows: {len(body_sample):,} body; {len(title_sample):,} title")
print("Same columns in both files:", body_sample.columns.equals(title_sample.columns))

,body_dtype,title_dtype
SOURCE_SUBREDDIT,str,str
TARGET_SUBREDDIT,str,str
POST_ID,str,str
TIMESTAMP,str,str
LINK_SENTIMENT,str,str
PROPERTIES,str,str


Preview rows: 1,000 body; 1,000 title
Same columns in both files: True


### Subreddit embeddings

The embedding CSV has no header. Each row contains a subreddit name followed by 300 numerical coordinates derived from user participation across communities. We name these coordinates `embedding_1` through `embedding_300`; individual coordinates have no assigned topic or sentiment meaning.

These vectors describe communities, whereas the hyperlink `PROPERTIES` describe source posts. Similar vectors reflect similar participation patterns, not necessarily agreement or friendship.


In [4]:
embedding_sample = load_embedding_sample(
    RAW_DATA_DIR / "web-redditEmbeddings-subreddits.csv"
)

# Show a few coordinates to keep the preview readable.
display(embedding_sample.iloc[:5, :6])
print(f"Preview rows: {len(embedding_sample):,}")
print(f"Embedding coordinates: {embedding_sample.shape[1] - 1}")
print("Subreddit dtype:", embedding_sample["subreddit"].dtype)
print("Coordinate dtypes:", embedding_sample.iloc[:, 1:].dtypes.unique())

,subreddit,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5
0,spiders,0.158972,0.285813,0.226329,-0.183338,-0.163159
1,askreddit,-0.499114,0.323983,-0.424809,-0.222705,-0.327201
2,globaloffensivetrade,-0.023145,-1.199374,1.661484,-1.025296,1.424670
3,fireteams,2.492506,-2.529917,-0.448484,-3.543441,-0.586122
4,funny,-0.819370,-0.865261,0.301753,0.018787,0.201029


Preview rows: 1,000
Embedding coordinates: 300
Subreddit dtype: str
Coordinate dtypes: [dtype('float64')]


## 3. Source-post features

The `PROPERTIES` field contains 86 comma-separated numbers in a fixed order. We expand it into numeric columns using the feature order in the [dataset documentation](https://snap.stanford.edu/data/soc-RedditHyperlinks.html).

- The first 18 features describe text length and writing style.
- The next three are VADER positive, negative and compound sentiment scores.
- The remaining 65 describe LIWC lexical categories, such as words associated with anger.

These features describe the **source post**, rather than the linked target post. Unlike embedding coordinates, they have documented meanings. Sentiment and lexical features are also related to how the link labels were constructed, so associations with those labels require careful interpretation.


In [5]:
from src.data.loaders import parse_properties

body_features = parse_properties(body_sample["PROPERTIES"])
title_features = parse_properties(title_sample["PROPERTIES"])

preview_features = [
    "n_characters", "n_words", "fraction_uppercase",
    "readability_ari", "vader_compound", "liwc_anger",
]
print("Body source-post features")
display(body_features[preview_features].head())
print("Title source-post features")
display(title_features[preview_features].head())
print(f"Parsed feature columns: {body_features.shape[1]} body; {title_features.shape[1]} title")


Body source-post features


,n_characters,n_words,fraction_uppercase,readability_ari,vader_compound,liwc_anger
0,345.0,57.0,0.086957,35.577895,0.1748,0.000000
1,101.0,14.0,0.049505,16.049286,0.5538,0.000000
2,85.0,10.0,0.082353,23.605000,0.0000,0.000000
3,1124.0,191.0,0.057829,12.654154,0.9672,0.000000
4,715.0,119.0,0.025175,16.786246,0.8906,0.008333


Title source-post features


,n_characters,n_words,fraction_uppercase,readability_ari,vader_compound,liwc_anger
0,25.0,4.0,0.440000,10.00750,0.0000,0.000000
1,100.0,16.0,0.080000,12.00750,0.4404,0.058824
2,100.0,16.0,0.080000,12.00750,0.4404,0.058824
3,49.0,8.0,0.265306,9.41875,0.0000,0.000000
4,14.0,2.0,0.428571,12.54000,0.0000,0.000000


Parsed feature columns: 86 body; 86 title


`n_characters` and `n_words` measure post length; `fraction_uppercase` measures the share of uppercase characters. `readability_ari` is an automated readability index. `vader_compound` summarizes text sentiment, while `liwc_anger` measures an anger-related lexical category. Neither score alone determines whether a reference is negative toward its target.

## 4. Complete hyperlink datasets

We read each TSV in chunks of 25,000 rows. All 86 source-post features are parsed in each chunk, but only ten selected measurements are retained in memory alongside the link columns. The original files remain unchanged.

Chunking limits temporary parsing memory; the returned link tables still contain every record and occupy memory proportional to the dataset size. Timestamps are converted to datetime values as recorded, without assigning a timezone. Labels are converted to integers. No date filtering, community-name normalization or duplicate removal is applied here.


In [6]:
from src.data.loaders import load_links

body_links, body_overview = load_links(
    RAW_DATA_DIR / "soc-redditHyperlinks-body.tsv"
)
title_links, title_overview = load_links(
    RAW_DATA_DIR / "soc-redditHyperlinks-title.tsv"
)

file_overview = pd.DataFrame([body_overview, title_overview]).set_index("file")
display(file_overview)
print(f"Total raw hyperlink records: {file_overview['rows'].sum():,}")


,rows,first_timestamp,last_timestamp,negative_links,nonnegative_links
file,,,,,
soc-redditHyperlinks-body.tsv,286561,2013-12-31 16:39:58,2017-04-30 16:58:21,21070,265491
soc-redditHyperlinks-title.tsv,571927,2013-12-31 16:20:20,2017-04-30 16:47:50,61140,510787


Total raw hyperlink records: 858,488


The counts describe complete files, unlike the earlier previews. Negative and nonnegative counts use the released labels; nonnegative includes both neutral and positive references. The two tables remain separate to preserve the distinction between body and title hyperlinks.


### Complete subreddit embeddings

We load all rows of the embedding CSV, preserving the subreddit names and the original 300 coordinates. The file has no header, so the first row is retained as data. The vectors are not normalized or filtered at this stage.


In [7]:
from src.data.loaders import load_embeddings

embeddings, embedding_overview = load_embeddings(
    RAW_DATA_DIR / "web-redditEmbeddings-subreddits.csv"
)

display(pd.DataFrame([embedding_overview]).set_index("file"))
display(embeddings.iloc[:5, :6])


,rows,coordinates
file,,
web-redditEmbeddings-subreddits.csv,51278,300


,subreddit,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5
0,spiders,0.158972,0.285813,0.226329,-0.183338,-0.163159
1,askreddit,-0.499114,0.323983,-0.424809,-0.222705,-0.327201
2,globaloffensivetrade,-0.023145,-1.199374,1.661484,-1.025296,1.424670
3,fireteams,2.492506,-2.529917,-0.448484,-3.543441,-0.586122
4,funny,-0.819370,-0.865261,0.301753,0.018787,0.201029


## 5. Data-quality checks

We check the complete link tables for missing or blank required fields, invalid timestamps and labels, repeated source–target–post keys, and self-links. The combined check also detects duplicate keys shared between the body and title files. Duplicate counts count occurrences after the first; names are compared as supplied.

Feature parsing has already checked that every record contains 86 finite numerical values. Invalid feature vectors or unexpected labels stop loading rather than being silently discarded.

For embeddings, we check missing names, duplicate names, nonfinite coordinates and zero vectors. A zero vector cannot support cosine similarity. These checks report issues without changing the data.


In [8]:
from src.data.validation import audit_links, audit_embeddings

link_quality = pd.DataFrame({
    "body": audit_links(body_links),
    "title": audit_links(title_links),
    "combined": audit_links(pd.concat([body_links, title_links], ignore_index=True)),
})
display(link_quality)

embedding_quality = audit_embeddings(embeddings)
display(embedding_quality.to_frame())


,body,title,combined
rows,286561,571927,858488
rows_with_missing_required_fields,0,0,0
missing_required_values,0,0,0
invalid_or_missing_timestamps,0,0,0
invalid_or_missing_labels,0,0,0
duplicate_source_target_post_keys,0,0,0
self_links,0,0,0


,count
rows,51278
coordinates,300
missing_or_blank_names,0
duplicate_names,0
nonfinite_vectors,0
zero_vectors,9


## 6. Embedding coverage

We match community names after stripping surrounding whitespace and converting to lowercase. An embedding is usable when its name is nonblank and all 300 coordinates are finite, with at least one nonzero value. The input tables are unchanged.

Community coverage counts distinct communities appearing at either end of a hyperlink. Link coverage distinguishes both endpoints covered, source only, target only and neither. We compare negative-link proportions across these groups and the activity of covered versus uncovered communities.

These measurements use all raw hyperlink dates, before analysis-period filtering. A community's activity is its number of incoming plus outgoing link appearances; a self-link contributes twice. Negative proportions are weighted by links, while the activity summaries give each community equal weight.


In [9]:
from src.utils.coverage import embedding_coverage

community_coverage, link_coverage, community_activity = embedding_coverage(
    pd.concat([body_links, title_links], ignore_index=True), embeddings
)
print("Community coverage and activity")
display(community_coverage)
print("Link coverage and negativity")
display(link_coverage)


Community coverage and activity


,communities,community_share,median_link_appearances,mean_link_appearances
coverage,,,,
usable_embedding,31238,0.46499,5.0,52.007107
no_usable_embedding,35942,0.53501,1.0,2.570196


Link coverage and negativity


,links,negative_links,link_share,negative_rate
coverage,,,,
both,768835,76797,0.895569,0.099887
source_only,23051,771,0.026851,0.033448
target_only,63877,4570,0.074406,0.071544
neither,2725,72,0.003174,0.026422


Community coverage and link coverage have different denominators. If covered communities account for a large share of activity, link coverage can be much higher than community coverage. Differences in negative-link proportions indicate that the similarity-analysis subset may differ from the complete hyperlink population; they do not show that embedding availability causes negativity.

Links without usable endpoint embeddings remain available for temporal analysis. Audience-similarity comparisons require usable embeddings at both endpoints.